If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec08_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 8: Distributions — Bar Charts, Binning, Histograms
v.ekc

A **distribution** answers: what values does a variable take, and how often? Today: categorical distributions (bar charts) and numerical ones (histograms) — plus the Area Principle that keeps them honest. (Slides: KL Lecture 08 deck.)

**Today's sections:**
1. Categorical distributions
2. Bar charts
3. Numerical distributions and binning
4. Histograms
5. The vertical axis

In [ ]:
from datascience import *
import numpy as np
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
plots.rcParams["patch.force_edgecolor"] = True

---
## 1. Categorical Distributions

Which studios released the top movies? Let's count — first the hard way, then the one-liner.

### Try It 1: Count the studios 😊

1. How many movies were released by MGM? Disney? Fox? (Use the moves you know.)

In [ ]:
top_movies = Table.read_table('top_movies_2017.csv')
top_movies

In [ ]:
studios = top_movies.select('Studio')
studios

In [ ]:
num_MGM = ...
num_MGM

In [ ]:
num_Disney = ...
num_Disney

In [ ]:
num_Fox = ...
num_Fox

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*`where` + `num_rows` works — once per studio. Keep reading for the better way.*

```python
# MGM
num_MGM = studios.where('Studio', 'MGM').num_rows

# Disney
num_Disney = studios.where('Studio', 'Disney').num_rows

# Fox
num_Fox = studios.where('Studio', 'Fox').num_rows
```

</details>

Let's explore another way to do this.

> **`group` counts every category at once** — one line instead of one `where` per studio. This method is about to become your best friend.

In [ ]:
studio_distribution = studios.group('Studio')
studio_distribution

In [ ]:
sum(studio_distribution.column('count'))

---
## 2. Bar Charts

A categorical distribution's natural picture. Sorting first makes it readable. (Why no pie charts? KL deck, slide 12 — comparing angles is hard; comparing lengths is easy. 🥧🚫)

In [ ]:
studio_distribution.barh('Studio')

In [ ]:
studio_distribution.sort('count', descending=True).barh('Studio')

---
## 3. Numerical Distributions and Binning

Numbers need a different move: chop the number line into **bins**, then count how many values land in each. (KL deck, slides 16–17.)

### 📋 Board Reference

| Code | What it does |
|---|---|
| `t.group('col')` | count occurrences of each category |
| `t.bin('col', bins=arr)` | count values landing in each bin |
| `t.hist('col', unit='Year')` | histogram, auto bins |
| `t.hist('col', bins=arr, unit='Year')` | histogram with your bins |
| `[a, b)` | bins include the left edge, not the right |

In [ ]:
ages = 2024 - top_movies.column('Year')
top_movies = top_movies.with_column('Age', ages)
top_movies

In [ ]:
top_movies.num_rows

In [ ]:
min(ages), max(ages)

In [ ]:
equal_bins = top_movies.bin('Age', bins = np.arange(0, 111, 10))
equal_bins.show()

In [ ]:
sum(equal_bins.column('Age count'))

In [ ]:
my_bins = make_array(0, 10, 20, 25, 40, 57, 60, 103)

In [ ]:
binned_data = top_movies.bin('Age', bins = my_bins)
binned_data

In [ ]:
sum(binned_data.column('Age count'))

### Try It 2: Bin detective 🔍

1. How many movies in `top_movies` have an age between 60 and 103 (excluding 103)? Does anything about the bin table seem strange to you?

In [ ]:
# 1. movies aged [60, 103)


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Each row of the bin table counts `[left edge, next edge)` — and the last row is just the right endpoint, so its count is always 0.*

```python
# 1.
binned_data.where('bin', 60).column('Age count').item(0)   # 33
```

*The "strange" parts: the bins aren't equally wide, and that final `103` row exists only to close the last bin.*

</details>

---
## 4. Histograms

A histogram is binning, drawn. But careful — with **unequal bins**, taller doesn't mean more. The **Area Principle** (KL deck, slides 18–21): the *area* of each bar shows the percent of data in that bin.

In [ ]:
# Let's explore the hist function!
top_movies.hist('Age', unit='Year')

In [ ]:
# Let's try equally spaced bins
top_movies.hist('Age', bins = np.arange(0, 111, 10), unit = 'Year')

In [ ]:
1.6 * (20 - 10)

In [ ]:
equal_bins

In [ ]:
total_count = sum(equal_bins.column('Age count'))
32/total_count

> **Height = density, area = percent.** The `[10, 20)` bar has height ≈ 1.6 percent-per-year × width 10 years = 16% of the movies. Read histogram heights as *crowding*, not counts.

---
## 5. The Vertical Axis

What happens if we plot raw counts on unequal bins instead? (`normed=False` — spoiler: don't.)

In [ ]:
my_bins

In [ ]:
binned_data

In [ ]:
# Add a column containing what percent of movies are in each bin
binned_data = binned_data.with_column(
    'Percent', 100*binned_data.column('Age count')/200)
binned_data

In [ ]:
# Let's make this histogram
top_movies.hist('Age', bins = my_bins, unit = 'Year', normed = False)

> **Compare this to the honest version above:** the wide `[40, 57)` bin *looks* huge on a count scale simply because it's wide. The Area Principle exists to stop exactly this lie. 😬

---
> **Today's takeaway:** `group` + bar chart for categories; `bin`/`hist` for numbers; and on histograms, area — not height — tells you how much data is where. Homework 3's histogram section tests this idea directly.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — distribution of a column
```python
t.group('category').sort('count', descending=True).barh('category')
t.hist('number', bins=np.arange(start, stop, width), unit='...')
```